<a name="top"></a><img src="images/chisel_1024.png" alt="Chisel logo" style="width:480px;" />

# Module 4.1: Introduction to FIRRTL
**Prev: [Generators: Types](3.6_types.ipynb)**<br>
**Next: [FIRRTL AST Traversal](4.2_firrtl_ast_traversal.ipynb)**

## Motivation
You've learned some Scala and written some Chisel, and for 90% of users, that should be enough to become a Chisel aficionado.

However, some use cases are better expressed as a programmatic analysis or transformation of a finished Chisel design, rather than as a generator.

For example, suppose we want to count the number of registers in a design, or the number of adders in every module of a hierarchy. This would be difficult to do as a generator, because a generator only sees the Scala objects it is building at that moment. Instead, we can look at the *elaborated* design, which is what the FIRRTL language describes.

In this module we will learn to read FIRRTL, and then write small Scala programs that parse, analyze and rewrite it.

## Setup
If a cell fails or hangs, see [Troubleshooting](docs/troubleshooting.md).

In [ ]:
interp.load.module(os.pwd / "source" / "load-ivy.sc")

In [ ]:
import chisel3._
import chisel3.Input // avoid a name clash with almond.input.Input
import chisel3.util._
import chisel3.reflect.DataMirror

## What is FIRRTL?
As you've probably become aware, when you execute a Chisel design, it elaborates (executes the surrounding Scala code) to construct an instance of your generator, with all Scala parameters resolved.

Chisel does not directly emit Verilog. Instead, it emits an intermediate representation called **FIRRTL** (Flexible Intermediate Representation for RTL), which represents the elaborated (parameter-resolved) RTL instance. The text that Chisel itself produces is sometimes called **CHIRRTL** ("Chisel FIRRTL"), because it still leaves some work, like width inference, to the compiler.

The compilation flow of Chisel 7 looks like this:

```
 Chisel (Scala)
      |  elaboration (running your generator)
      v
 CHIRRTL text            <- human readable; what this notebook is about
      |  ----------------------------------------- firtool (CIRCT) ----
      |  parse, infer widths, check, optimize
      v
 FIRRTL dialect (MLIR)
      |  lower types, remove `when`, ...
      v
 HW dialect (MLIR)
      |  emit
      v
 SystemVerilog
```

Everything below the CHIRRTL line is done by [CIRCT](https://circt.llvm.org)'s `firtool`, a native program that Chisel downloads and runs for you. In older versions of Chisel (3.x and before), the same job was done by a FIRRTL compiler written in Scala, and a whole chapter of this bootcamp was spent on writing Scala "transforms" for it. That Scala library no longer exists: the FIRRTL *language* is the contract between Chisel and the compiler, and any tool that can read the text can work with it.

That is also good news for us. FIRRTL text is simple, line-oriented and well specified, so with a few dozen lines of Scala we can build our own data structure (an abstract-syntax-tree, or AST) from it and analyze it. This is what Modules 4.2 to 4.4 do.

Let's take a look! We will take a simple Chisel design, elaborate it, and inspect what FIRRTL it generates.

First, we define a Chisel module, which delays its input signal by two cycles.

In [ ]:
class DelayBy2(width: Int) extends Module {
  val io = IO(new Bundle {
    val in  = Input(UInt(width.W))
    val out = Output(UInt(width.W))
  })
  val r0 = RegNext(io.in)
  val r1 = RegNext(r0)
  io.out := r1
}

Next, let's elaborate it and print the FIRRTL it generates. `getFirrtl` is a helper of this bootcamp (see `source/load-ivy.sc`); it is a thin wrapper around `circt.stage.ChiselStage.emitCHIRRTL`.

In [ ]:
println(getFirrtl(new DelayBy2(4)))

As you can see, the serialized FIRRTL looks very similar to what our Chisel design looks like, with all generator parameters resolved. Let's go through it from top to bottom:

* `FIRRTL version 7.0.0` is the version of the language specification.
* `circuit DelayBy2 :` starts the circuit, whose name is its top module. Everything after the `%[[` up to the matching `]]` is a JSON list of *annotations*: extra information for the compiler that is not part of the circuit itself. Here, Chisel only says which modules are candidates for de-duplication (`DedupGroupAnnotation`). Libraries and user code can add their own.
* The `layer` declarations (`Verification`, `Assert`, ...) declare optional pieces of hardware, which can be switched on or off at build time. Chisel always declares the standard ones for verification code such as `printf`, `assert`, `assume` and `cover`. They are not relevant for our design.
* `public module DelayBy2 :` is a module definition. Only the top module is `public`: it is the one visible from the outside.
* `input clock : Clock` and `input reset : UInt<1>` are the implicit clock and reset ports that every Chisel `Module` has. Note that a Chisel `Bundle` is *not* flattened, it is a single port with an aggregate type, `output io : { flip in : UInt<4>, out : UInt<4>}`. A `flip` field has the opposite direction of the port, so `io.in` is an input.
* `reg r0 : UInt, clock` declares a register named after the Scala `val` that holds it. Its type is just `UInt`: the width is unknown, to be inferred by the compiler (from `io.in` in this case). A register with a reset value would be declared as `regreset`.
* `connect r0, io.in` assigns a source to a sink. It is the only kind of assignment in FIRRTL, and, as in Chisel, a later connect overrides an earlier one (last-connect semantics).
* `@[cmd3.sc 6:19]` at the end of a line is a *source locator*: it records the Scala file, line and column that created this line of FIRRTL. This is how tools can report errors in terms of your Chisel code. (Here the file is the notebook cell that was executed.)

### Following the pipeline
We can ask Chisel to stop the pipeline at the later stages as well. `ChiselStage` has an `emit...` function for each of them. Note that you have to write `_root_.circt` because `chisel3.util` contains another package named `circt`, which our `import chisel3.util._` makes visible.

First, the FIRRTL dialect. This is the CHIRRTL after `firtool` has parsed it, checked it and inferred all widths. It is written in MLIR syntax (we only print the interesting module, not the layer declarations):

In [ ]:
val firrtlDialect = _root_.circt.stage.ChiselStage.emitFIRRTLDialect(new DelayBy2(4))
println(
  firrtlDialect.linesIterator.dropWhile(!_.contains("firrtl.module")).takeWhile(!_.startsWith("  }")).mkString("\n")
)

Notice that the widths are known now (`!firrtl.uint<4>` for `r0`), that the bundle is now a list of scalar ports `%io_in` and `%io_out`, and that the output port is driven through a helper wire.

Next is the HW dialect, where the design is described with hardware primitives (`seq.firreg` is a register) instead of FIRRTL's types:

In [ ]:
val hwDialect = _root_.circt.stage.ChiselStage.emitHWDialect(new DelayBy2(4))
println(hwDialect.linesIterator.dropWhile(!_.contains("hw.module")).takeWhile(!_.startsWith("  }")).mkString("\n"))

Finally, SystemVerilog, which is what `getVerilog` returns. `emitSystemVerilog` also accepts options for `firtool`, which is how you control the generated code. For example, the options below remove the debug information and the random initialization of registers (which firtool normally emits for simulation). `firtool --help` lists all of them. The output also contains some extra files for the verification layers, which we cut off:

In [ ]:
val sv = _root_.circt.stage.ChiselStage.emitSystemVerilog(
  new DelayBy2(4),
  firtoolOpts = Array("--disable-all-randomization", "--strip-debug-info")
)
println(sv.split("// ----- 8<")(0))

### Hierarchy
Modules that instantiate other modules show up as `inst` statements. Let's define a design with a module `Top` that instantiates `AddMe` twice, with different parameters:

In [ ]:
class AddMe(val nInputs: Int, val width: Int) extends Module {
  val io = IO(new Bundle {
    val in  = Input(Vec(nInputs, UInt(width.W)))
    val out = Output(UInt(width.W))
  })
  io.out := io.in.reduce(_ +& _)
}

class Top extends Module {
  val io = IO(new Bundle {
    val a = Input(UInt(4.W))
    val b = Input(UInt(4.W))
    val y = Output(UInt(5.W))
    val z = Output(UInt(5.W))
  })
  val m0 = Module(new AddMe(2, 4))
  val m1 = Module(new AddMe(3, 4))
  m0.io.in(0) := io.a
  m0.io.in(1) := io.b
  m1.io.in(0) := io.a
  m1.io.in(1) := io.b
  m1.io.in(2) := m0.io.out
  io.y        := m0.io.out
  io.z        := m1.io.out
}

In [ ]:
// Skip the annotations and layers, and show only the modules
println(getFirrtl(new Top).linesIterator.dropWhile(!_.contains("module")).mkString("\n"))

A few things to observe:
* A circuit is a flat list of modules, and *modules are never nested*. `Top` refers to the other two by name with `inst m0 of AddMe`.
* The two instances have different parameters, so they are different modules, which Chisel names `AddMe` and `AddMe_1`. Two instances with identical parameters would share one module definition.
* Clock and reset are connected explicitly (`connect m0.clock, clock`), as is everything else. FIRRTL has no implicit wiring.
* Ports of an instance are accessed like fields of a bundle: `m0.io.in[0]`, `m0.io.out`.
* The module `Top` is last, and it is the only `public` one.

### Looking at the design from Chisel
The FIRRTL text is not the only way to look at a design. While Chisel is elaborating, `chisel3.reflect.DataMirror` can tell you about the `Data` objects you hold: whether they are ports, wires or registers, their direction and their width. We use it below to list the ports of a submodule from within its parent. Note that this information is available only *after* a module has been closed (`Module(new AddMe(...))` has returned), because Chisel only then knows all of its ports.

In [ ]:
class Parent extends Module {
  val io = IO(new Bundle { val in = Input(UInt(4.W)); val out = Output(UInt(5.W)) })
  val r  = RegInit(0.U(3.W))
  val w  = Wire(UInt(3.W))
  w := r
  println(s"r: isReg=${DataMirror.isReg(r)} isWire=${DataMirror.isWire(r)}")
  println(s"w: isReg=${DataMirror.isReg(w)} isWire=${DataMirror.isWire(w)}")

  val child = Module(new AddMe(2, 4))
  for ((name, port) <- DataMirror.fullModulePorts(child)) {
    println(f"child port $name%-8s ${DataMirror.directionOf(port)}%-14s width=${port.widthOption}")
  }
  child.io.in(0) := io.in
  child.io.in(1) := io.in
  io.out         := child.io.out
}
val parentFirrtl = getFirrtl(new Parent)

`DataMirror` is handy inside generators, but it cannot see everything: for example, it cannot enumerate the registers or the operators of a module, and it works only while elaborating. The FIRRTL text, on the other hand, contains the *whole* design, so we will focus on it in the rest of this module.

# FIRRTL Language Overview

This section describes the most common elements of the FIRRTL language. For more detail on components not mentioned here, please refer to [The FIRRTL Specification](https://github.com/chipsalliance/firrtl-spec).

## Circuit
The circuit is the root of any FIRRTL file. There is only ever one circuit, and it contains a list of module definitions and the name of the top-level module.
```
circuit Adder :
  ... // list of modules
```

## Module
Modules are the unit of modularity within FIRRTL and are never directly nested (declaring an instance of a module is a statement). Each module has a name, a list of ports, and a body containing its implementation.
```
module Adder :
  ... // list of ports
  ... // statements
```

## Port
A port defines part of a module's interface, and has a name, a direction (`input` or `output`), and a type. Types are:
```
input x : UInt<8>                    // unsigned integer of 8 bits; `UInt` alone means "infer the width"
input s : SInt<8>                    // signed integer
input c : Clock                      // clock, `Reset` and `AsyncReset` are the reset types
output v : UInt<4>[2]                // vector (array) of 2 elements
output io : { flip in : UInt<4>, out : UInt<4> } // bundle (struct); `flip` reverses the direction of a field
```

## Statements
A statement describes the components within a module and how they interact. The most common ones:
```
wire w : UInt<4>                                // wire, can be both a source and a sink
reg r : UInt<4>, clock                          // register without reset
regreset r : UInt<4>, clock, reset, UInt<4>(0h0) // register with synchronous reset, and its reset value
node n = add(x, y)                              // named expression (immutable)
inst m0 of Adder                                // instance of another module
connect w, n                                    // connection, with last-connect semantics
when cond :                                     // conditional; may be followed by `else :`
  ...
invalidate w                                    // "don't care" value
printf(...), assert(...)                        // verification statements
cmem / smem                                     // memories
```
Every `when` block is indented. Chisel's `when`/`.elsewhen`/`.otherwise` become nested `when`/`else` blocks.

## Expressions
Expressions represent references to declared components or logical and arithmetic operations. The most common ones:
```
UInt<4>(0h5)        // literal: type, then value in hex (0h5), or decimal
x                   // reference to a port, wire, register, node or instance
io.in               // sub-field of a bundle, also `m0.io.out` for instance ports
v[1]                // sub-index of a vector
mux(sel, a, b)      // multiplexer
add(a, b)           // primitive operation (primop)
bits(a, 3, 0)       // primops can take integer constants after their arguments
```
There are only a few dozen primitive operations. Among them: `add`, `sub`, `mul`, `div`, `rem`, `lt`, `leq`, `gt`, `geq`, `eq`, `neq`, `and`, `or`, `xor`, `not`, `shl`, `shr`, `cat`, `bits`, `head`, `tail`, `pad`, `asUInt`, `asSInt`, `andr`, `orr`, `xorr`. Note that FIRRTL's `add` has a result that is one bit wider than its operands, just like Chisel's `+&`. A Chisel `+` is a `tail(add(a, b), 1)`, which drops the carry bit. We will see this in the next sections.

<span style="color:red">**Exercise: Read the FIRRTL**</span><br>
Below is a module `EnabledReg`, which should keep its value unless `io.en` is true, in which case it loads `io.d`. The register must reset to 0. Complete the module, and then look at the FIRRTL printed by the next line. Which FIRRTL statements did your Chisel code turn into? (Hint: you should find a `regreset`, a `when` and some `connect`s.)

In [ ]:
class EnabledReg extends Module {
  val io = IO(new Bundle {
    val en = Input(Bool())
    val d  = Input(UInt(8.W))
    val q  = Output(UInt(8.W))
  })

  val r = ??? // a register with an 8-bit width and reset value 0
  ??? // when `en` is true, load `d` into r
  io.q := r
}

val enabledRegFirrtl = getFirrtl(new EnabledReg)
println(enabledRegFirrtl.linesIterator.dropWhile(!_.contains("module")).mkString("\n"))
assert(enabledRegFirrtl.contains("regreset") && enabledRegFirrtl.contains("when io.en"))
println("SUCCESS!!")

<div id="container"><section id="accordion"><div>
<input type="checkbox" id="check-1" />
<label for="check-1"><strong>Solution</strong> (click to toggle displaying)</label>
<article>
<pre style="background-color:#f7f7f7">
class EnabledReg extends Module {
  val io = IO(new Bundle {
    val en = Input(Bool())
    val d  = Input(UInt(8.W))
    val q  = Output(UInt(8.W))
  })

  val r = RegInit(0.U(8.W))
  when (io.en) {
    r := io.d
  }
  io.q := r
}
</pre></article></div></section></div>

That's it for this section! In the next section, we will build a Scala data structure for FIRRTL and look at how a program walks it.

# You're done!

[Return to the top.](#top)